# Week 6: Agentic RAG with LangGraph

## Overview

이 노트북은 5주차 Hybrid+Rerank 검색 전략을 기반으로 LangGraph StateGraph를 활용한 Agentic RAG를 구현한다.

**4개 노드 구조:**
1. `retrieve`: Hybrid+Rerank 검색 (metadata_filter 적용 가능)
2. `grade_documents`: LLM Judge로 관련성 판단
3. `rewrite_query`: Self-Query (metadata 추출) 또는 Query 재작성
4. `generate`: RAG 답변 생성

**Routing:**
- START → retrieve → grade_documents
- relevant → generate → END
- not_relevant (retry < 2) → rewrite_query → retrieve
- not_relevant (retry >= 2) → cannot_answer → END

In [2]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI

from src.agent import (
    GraphState,
    make_filterable_hybrid_retriever,
    build_agent_graph,
    run_agent,
)

/Users/joyoungha/Desktop/project/rag-agent-portfolio/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. State 정의 (§4.1)

In [3]:
# GraphState moved to src/agent.py (see imports in cell 1).

## 2. Retriever 로드 (5주차 Hybrid+Rerank)

In [4]:
from src.vectorstore import load_vectorstore
from src.retrieval import (
    HybridRetrieverConfig,
    RerankConfig,
    create_hybrid_retriever,
    create_reranker,
    rerank_documents,
    extract_documents_from_vectorstore,
)
from langchain_community.retrievers import BM25Retriever

CHROMA_DIR = Path("../data/chroma_db_c3")

print("Loading vectorstore...")
vectorstore = load_vectorstore(CHROMA_DIR, collection_name="lg_manuals_c3")
print(f"Loaded {vectorstore._collection.count()} documents")

# BM25용 문서 추출
all_docs = extract_documents_from_vectorstore(vectorstore)
print(f"Extracted {len(all_docs)} documents for BM25")

# Reranker 로드
print("Loading reranker model...")
reranker = create_reranker("dragonkue/bge-reranker-v2-m3-ko")
print("Reranker loaded.")

Loading vectorstore...


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


Loaded 258 documents
Extracted 258 documents for BM25
Loading reranker model...


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 7141.02it/s]


Reranker loaded.


## 3. LLM 설정

In [5]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)

## 4. 노드 구현 (§4.2)

### 4.1 retrieve 노드

In [6]:
# retrieve_node moved to src/agent.py (encapsulated by make_filterable_hybrid_retriever + build_agent_graph).

### 4.2 grade_documents 노드 — LLM Judge

In [7]:
# grade_documents_node + GRADE_PROMPT moved to src/agent.py.

### 4.3 rewrite_query 노드 — Self-Query 통합 (§5)

In [8]:
# rewrite_query_node + SELF_QUERY_PROMPT + REWRITE_PROMPT moved to src/agent.py.

### 4.4 generate 노드

In [9]:
# generate_node + GENERATE_PROMPT moved to src/agent.py.

### 4.5 cannot_answer 노드

In [10]:
# cannot_answer_node moved to src/agent.py.

## 5. 조건부 엣지 (Routing) (§4.3)

In [11]:
# route_after_grade moved to src/agent.py.

## 6. StateGraph 구성

In [12]:
retriever_fn = make_filterable_hybrid_retriever(
    vectorstore=vectorstore,
    all_docs=all_docs,
    reranker=reranker,
)
app = build_agent_graph(retriever_fn, llm)
print("Graph compiled successfully!")

Graph compiled successfully!


## 7. Workflow Diagram 출력 (§6)

In [13]:
# Mermaid diagram
try:
    mermaid_code = app.get_graph().draw_mermaid()
    print("=== Mermaid Diagram ===")
    print(mermaid_code)
    
    # 파일로 저장
    with open("../docs/week6_workflow_diagram.md", "w") as f:
        f.write("# Week 6 Agentic RAG Workflow Diagram\n\n")
        f.write("```mermaid\n")
        f.write(mermaid_code)
        f.write("\n```\n\n")
        f.write("## 노드 설명\n\n")
        f.write("- **retrieve**: Hybrid+Rerank 검색 (5주차 전략). metadata_filter 적용 가능\n")
        f.write("- **grade_documents**: LLM Judge로 관련성 판단 (relevant/not_relevant)\n")
        f.write("- **rewrite_query**: Self-Query (카테고리 추출) 또는 Query 재작성 (키워드 추가)\n")
        f.write("- **generate**: RAG 답변 생성\n")
        f.write("- **cannot_answer**: 2회 retry 후 답변 거절\n")
    print("\nDiagram saved to docs/week6_workflow_diagram.md")
except Exception as e:
    print(f"Mermaid generation failed: {e}")

=== Mermaid Diagram ===
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	retrieve(retrieve)
	grade_documents(grade_documents)
	rewrite_query(rewrite_query)
	generate(generate)
	cannot_answer(cannot_answer)
	__end__([<p>__end__</p>]):::last
	__start__ --> retrieve;
	grade_documents -.-> cannot_answer;
	grade_documents -.-> generate;
	grade_documents -.-> rewrite_query;
	retrieve --> grade_documents;
	rewrite_query --> retrieve;
	cannot_answer --> __end__;
	generate --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc


Diagram saved to docs/week6_workflow_diagram.md


## 8. 단일 질문 테스트

In [14]:
# run_agentic_rag moved to src/agent.py as run_agent(graph, question).

In [15]:
# Q20 테스트 (공기청정기 소음 - Self-Query가 필요한 케이스)
test_question = "공기청정기 소음이 심해요"
result = run_agent(app, test_question)

print(f"질문: {test_question}")
print(f"\n라우팅 기록: {result['route_history']}")
print(f"메타데이터 필터: {result['metadata_filter']}")
print(f"재시도 횟수: {result['retry_count']}")
print(f"\n검색된 문서 카테고리: {[d.metadata.get('category') for d in result['documents']]}")
print(f"\n답변:\n{result['answer'][:500]}...")
print(f"\nLatency breakdown: {result['latency_breakdown']}")
print(f"Total latency: {result['total_latency']:.2f}s")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


질문: 공기청정기 소음이 심해요

라우팅 기록: ['retrieve(filter=None)', 'grade(not_relevant)', 'self_query(category=airpurifier)', "retrieve(filter={'category': 'airpurifier'})", 'grade(not_relevant)', "rewrite('LG 공기청정기 소음 문제 해결 방법은 무엇인가요?...')", "retrieve(filter={'category': 'airpurifier'})", 'grade(not_relevant)', 'cannot_answer']
메타데이터 필터: {'category': 'airpurifier'}
재시도 횟수: 2

검색된 문서 카테고리: ['airpurifier', 'airpurifier', 'airpurifier', 'airpurifier', 'airpurifier']

답변:
죄송합니다. 제공된 매뉴얼에서 해당 질문에 대한 정보를 찾을 수 없습니다....

Latency breakdown: {'retrieve': 38.96356701850891, 'grade': 4.387524366378784, 'rewrite': 1.9246509075164795}
Total latency: 45.29s


## 9. 전체 평가 (23 Questions)

In [16]:
from src.evaluation import load_eval_questions

EVAL_PATH = Path("../data/eval_questions_v2.json")
questions = load_eval_questions(EVAL_PATH)
print(f"Loaded {len(questions)} evaluation questions")

Loaded 23 evaluation questions


In [17]:
# 전체 평가 실행
results = []

for i, q in enumerate(questions):
    print(f"[{i+1}/{len(questions)}] {q.question[:40]}...", end=" ")

    result = run_agent(app, q.question)

    # Top-1 category 추출
    top1_category = result["documents"][0].metadata.get("category", "unknown") if result["documents"] else "none"
    is_correct = top1_category == q.category

    results.append({
        "id": q.id,
        "question": q.question,
        "expected": q.category,
        "predicted": top1_category,
        "correct": is_correct,
        "retry_count": result["retry_count"],
        "route_history": result["route_history"],
        "metadata_filter": result["metadata_filter"],
        "total_latency": result["total_latency"],
        "latency_breakdown": result["latency_breakdown"],
    })

    status = "O" if is_correct else "X"
    print(f"{status} (retry={result['retry_count']}, {result['total_latency']:.1f}s)")

print("\nEvaluation complete!")

[1/23] 정수기 필터 교체 주기는 얼마인가요?... O (retry=0, 13.6s)
[2/23] WD523A 모델의 제어창 사용법을 알려주세요... O (retry=0, 17.1s)
[3/23] 물맛이 이상할 때 어떻게 해야 하나요?... O (retry=0, 22.7s)
[4/23] 정수기 출수구 살균 기능은 어떻게 사용하나요?... O (retry=0, 14.4s)
[5/23] 온수 잠금 기능을 설정하는 방법... O (retry=0, 18.7s)
[6/23] AS281DAW 필터 수명은 얼마나 되나요?... O (retry=1, 28.1s)
[7/23] 공기청정기 필터 청소는 어떻게 하나요?... O (retry=2, 43.6s)
[8/23] 공기가 탁할 때 어떤 모드를 사용해야 하나요?... O (retry=0, 18.4s)
[9/23] 공기청정기 센서 청소 방법을 알려주세요... O (retry=0, 20.2s)
[10/23] 상태 표시등이 빨간색일 때 무슨 의미인가요?... O (retry=0, 22.1s)
[11/23] 청소기 배터리 충전 시간은 몇 시간인가요?... O (retry=0, 23.3s)
[12/23] 배터리가 빨리 닳아요... O (retry=0, 27.9s)
[13/23] 먼지 분리기 청소는 어떻게 하나요?... O (retry=0, 27.9s)
[14/23] 흡입구에 뭔가 걸렸을 때 어떻게 하나요?... O (retry=0, 21.2s)
[15/23] 보조 배터리 충전하는 방법... O (retry=0, 29.3s)
[16/23] LG ThinQ 앱 연결 방법을 알려주세요... O (retry=0, 19.0s)
[17/23] 와이파이 연결이 안 될 때... X (retry=0, 26.8s)
[18/23] 청소기 흡입력이 약해졌어요... O (retry=0, 28.7s)
[19/23] 필터 교체 후 해야 할 일이 있나요?... O (retry=0, 24.5s)
[20/23] 공기청정기 소음이 심해요... O (retry=2, 

In [18]:
# 결과 분석
import pandas as pd

df = pd.DataFrame(results)

# 전체 정확도 (23문항)
accuracy_23 = df["correct"].mean()
correct_23 = df["correct"].sum()

# Q17 제외 정확도 (22문항)
df_22 = df[df["id"] != "Q17"]
accuracy_22 = df_22["correct"].mean()
correct_22 = df_22["correct"].sum()

# 평균 latency
avg_latency = df["total_latency"].mean()

# Retry 통계
retry_stats = df["retry_count"].value_counts().sort_index()

print("=== Agentic RAG 결과 ===")
print(f"\nTop-1 Accuracy (23q): {accuracy_23:.1%} ({correct_23}/23)")
print(f"Top-1 Accuracy (22q, Q17 제외): {accuracy_22:.1%} ({correct_22}/22)")
print(f"\n평균 Latency: {avg_latency:.2f}s")
print(f"\nRetry 통계:")
for retry_count, count in retry_stats.items():
    print(f"  retry={retry_count}: {count}건")

=== Agentic RAG 결과 ===

Top-1 Accuracy (23q): 95.7% (22/23)
Top-1 Accuracy (22q, Q17 제외): 100.0% (22/22)

평균 Latency: 27.01s

Retry 통계:
  retry=0: 19건
  retry=1: 1건
  retry=2: 3건


In [19]:
df.head()

,id,question,expected,predicted,correct,retry_count,route_history,metadata_filter,total_latency,latency_breakdown
0,Q01,정수기 필터 교체 주기는 얼마인가요?,waterpurifier,waterpurifier,True,0,"[retrieve(filter=None), grade(relevant), gener...",None,13.579448,"{'retrieve': 10.912721872329712, 'grade': 0.69..."
1,Q02,WD523A 모델의 제어창 사용법을 알려주세요,waterpurifier,waterpurifier,True,0,"[retrieve(filter=None), grade(relevant), gener...",None,17.071528,"{'retrieve': 11.219921112060547, 'grade': 0.75..."
2,Q03,물맛이 이상할 때 어떻게 해야 하나요?,waterpurifier,waterpurifier,True,0,"[retrieve(filter=None), grade(relevant), gener...",None,22.731549,"{'retrieve': 18.111834049224854, 'grade': 0.59..."
3,Q04,정수기 출수구 살균 기능은 어떻게 사용하나요?,waterpurifier,waterpurifier,True,0,"[retrieve(filter=None), grade(relevant), gener...",None,14.446055,"{'retrieve': 11.05093502998352, 'grade': 0.713..."
4,Q05,온수 잠금 기능을 설정하는 방법,waterpurifier,waterpurifier,True,0,"[retrieve(filter=None), grade(relevant), gener...",None,18.725956,"{'retrieve': 16.52459979057312, 'grade': 0.619..."


In [20]:
# 실패 케이스 상세
failures = df[~df["correct"]]

print("=== 실패 케이스 ===")
for _, row in failures.iterrows():
    print(f"\n[{row['id']}] {row['question']}")
    print(f"  Expected: {row['expected']}, Got: {row['predicted']}")
    print(f"  Route: {row['route_history']}")
    print(f"  Filter: {row['metadata_filter']}")

=== 실패 케이스 ===

[Q17] 와이파이 연결이 안 될 때
  Expected: waterpurifier, Got: vacuumcleaner
  Route: ['retrieve(filter=None)', 'grade(relevant)', 'generate']
  Filter: None


In [21]:
# 결과 저장
df.to_json("../data/week6_agentic_rag_results.json", orient="records", force_ascii=False, indent=2)
print("Results saved to data/week6_agentic_rag_results.json")

Results saved to data/week6_agentic_rag_results.json


## 10. Baseline 비교

In [22]:
# 5주차 Baseline vs Agentic RAG 비교표
print("=== Baseline vs Agentic RAG ===")
print()
print("| 구성 | Top-1 (22q) | Top-1 (23q) | 평균 Latency |")
print("|---|---|---|---|")
print(f"| Baseline (Hybrid+Rerank) | 90.9% (20/22) | 91.3% (21/23) | 6.73s |")
print(f"| Agentic RAG | {accuracy_22:.1%} ({correct_22}/22) | {accuracy_23:.1%} ({correct_23}/23) | {avg_latency:.2f}s |")

=== Baseline vs Agentic RAG ===

| 구성 | Top-1 (22q) | Top-1 (23q) | 평균 Latency |
|---|---|---|---|
| Baseline (Hybrid+Rerank) | 90.9% (20/22) | 91.3% (21/23) | 6.73s |
| Agentic RAG | 100.0% (22/22) | 95.7% (22/23) | 27.01s |
